# 11b - LLM report labels with model-adapt

Compares an instruction-tuned LLM against the rule-based extractor of notebook 11 on
the 58 expert-labelled studies, using my **model-adapt** library unchanged: JSONL
dataset -> `label_classification` prompts -> Hugging Face inference -> predictions,
metrics and a manifest pinning the model revision and dataset hash.

Each (study, target) pair is one example (58 x 12 = 696 prompts). Labels:
`positive`, `negative`, `not_mentioned`.

**Settings:** GPU (T4) and internet ON (installs model-adapt from GitHub and
downloads `Qwen/Qwen2.5-1.5B-Instruct`, about 3 GB, at a pinned revision).
**Inputs:** competition `train.csv`; notebook 11 output `weak_labels_v1.csv`.
**Outputs:** model-adapt run directory, `llm_vs_regex_v1.csv`.
Scaling to all 4,407 studies (about 53k prompts) is worth it only if the LLM clearly
beats the regex extractor here.

In [ ]:
%env GIT_TERMINAL_PROMPT=0
%pip install -q "model-adapt @ git+https://github.com/Z-bros/model-adapt-overcome.git"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# The competition mount path has differed between Kaggle runtimes; take the first that has train.csv.
_SLUG = "rsna-knee-abnormality-detection"
_roots = [Path("/kaggle/input/competitions") / _SLUG, Path("/kaggle/input") / _SLUG,
          *sorted(Path("/kaggle/input").glob("*/" + _SLUG)), *sorted(Path("/kaggle/input").glob("*"))]
ROOT = next((r for r in _roots if (r / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("competition train.csv not found; /kaggle/input has: "
                            + str(sorted(str(p) for p in Path("/kaggle/input").glob("*/*"))[:50]))
print("competition data:", ROOT)
OUT = Path("/kaggle/working")
TARGET_COLUMNS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 42
from pathlib import Path


def find_input(name):
    """Locate one attached upstream file by name.

    Only notebook outputs and datasets are searched: walking the competition
    mount would visit millions of DICOM files.
    """
    hits = []
    for base in (Path("/kaggle/input/notebooks"), Path("/kaggle/input/datasets"),
                 Path("/kaggle/input")):
        if base.is_dir():
            for child in base.iterdir():
                if (child.is_dir()
                        and child.name not in ("competitions", "notebooks", "datasets")
                        and not (child / "train_series").exists()
                        and not (child / "test_series").exists()):
                    hits += list(child.rglob(name))
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly one attached '{name}', found {len(hits)}: {hits[:5]}. "
            "Attach the upstream notebook output via Add Input."
        )
    return hits[0]

import json
import subprocess

import torch
import transformers
from model_adapt.core import load_config
from model_adapt.data import file_sha256

assert torch.cuda.is_available(), "enable a GPU accelerator"
print("transformers", transformers.__version__, "| GPU", torch.cuda.get_device_name(0))

WORK = OUT / "llm"
WORK.mkdir(parents=True, exist_ok=True)
QWEN_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
QUESTIONS = {
    "ACL": "anterior cruciate ligament (ACL) tear (high-grade partial or complete)",
    "MCL": "medial collateral ligament (MCL) tear (high-grade partial or complete)",
    "Medial Meniscus": "medial meniscus tear",
    "Lateral Meniscus": "lateral meniscus tear",
    "Medial OA": "medial tibiofemoral compartment osteoarthritis (substantial cartilage loss)",
    "Lateral OA": "lateral tibiofemoral compartment osteoarthritis (substantial cartilage loss)",
    "PF OA": "patellofemoral osteoarthritis (substantial cartilage loss)",
    "Effusion": "joint effusion",
    "Synovitis": "synovitis",
    "Baker's": "Baker's (popliteal) cyst",
    "Contusion": "bone contusion / bone bruise (traumatic bone marrow edema)",
    "Fracture": "fracture"
}
assert list(QUESTIONS) == TARGET_COLUMNS

## 1. Build the JSONL dataset (expert-labelled studies only)

In [ ]:
train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
gold = train[train[TARGET_COLUMNS].notna().all(axis=1)].sort_values("StudyInstanceUID")
assert len(gold) == 58

with (WORK / "llm_gold.jsonl").open("w", encoding="utf-8") as fh:
    for row in gold.to_dict("records"):
        report = " ".join(str(row["Report"]).split())
        for t in TARGET_COLUMNS:
            label = int(row[t])
            fh.write(json.dumps({
                "id": f"{row['StudyInstanceUID']}::{t}",
                "input": f"Target finding: {QUESTIONS[t]}\n\nKnee MRI report:\n{report}",
                "target": "positive" if label == 1 else "negative",
            }, ensure_ascii=False) + "\n")

sha = file_sha256(WORK / "llm_gold.jsonl")
print("examples:", sum(1 for _ in open(WORK / "llm_gold.jsonl")), "| sha256:", sha[:12])

## 2. Experiment config (validated by model-adapt before running)

In [ ]:
INSTRUCTION = (
    "You read knee MRI radiology reports, which may be written in any language. "
    "Decide whether the report says the target finding is present (positive), "
    "says it is absent or the structure is normal (negative), or does not address it "
    "(not_mentioned)."
)
(WORK / "llm_gold.toml").write_text(f"""
name = "rsna-llm-gold-qwen2.5-1.5b"
seed = {SEED}
output_dir = "outputs/rsna-llm-gold"

[task]
name = "label_classification"
labels = ["positive", "negative", "not_mentioned"]
instruction = {json.dumps(INSTRUCTION)}

[dataset]
source = "llm_gold.jsonl"
split = "train"
revision = "{sha}"

[model]
name_or_path = "Qwen/Qwen2.5-1.5B-Instruct"
revision = "{QWEN_REVISION}"
device = "cuda"
max_new_tokens = 8
batch_size = 8

[evaluation]
metrics = ["accuracy", "valid_rate", "n_examples", "n_labeled", "n_invalid"]
""", encoding="utf-8")
cfg = load_config(WORK / "llm_gold.toml")
print(cfg.name, cfg.model.name_or_path, cfg.model.revision[:12])

## 3. Run

In [ ]:
proc = subprocess.run(["model-adapt", "run", "llm_gold.toml"], cwd=WORK,
                      capture_output=True, text=True)
print(proc.stdout[-2000:])
if proc.returncode != 0:
    raise RuntimeError(proc.stderr[-2000:])
run_dir = sorted((WORK / "outputs" / "rsna-llm-gold").iterdir())[-1]
print("run dir:", run_dir)
print(json.loads((run_dir / "metrics.json").read_text()))

## 4. Compare with the regex extractor

LLM labels map to scores positive 0.9, not_mentioned 0.3, negative 0.1, invalid 0.3
(close to the extractor's silent value 0.28), so both are ranked by the same AUC.

In [ ]:
from sklearn.metrics import roc_auc_score


def safe_auc(y, p):
    """AUC, or NaN when only one class is present (the metric is undefined)."""
    y = np.asarray(y)
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan


def auc_table(gold, scores, n_boot=2000, seed=0):
    """Per-target AUC with a percentile bootstrap interval over studies."""
    rng = np.random.default_rng(seed)
    rows = []
    for t in TARGET_COLUMNS:
        y, p = gold[t].to_numpy(), scores[t].to_numpy()
        boots = []
        for _ in range(n_boot):
            i = rng.integers(0, len(y), len(y))
            a = safe_auc(y[i], p[i])
            if np.isfinite(a):
                boots.append(a)
        rows.append({"target": t, "auc": safe_auc(y, p),
                     "ci_low": np.percentile(boots, 2.5) if boots else np.nan,
                     "ci_high": np.percentile(boots, 97.5) if boots else np.nan,
                     "n_pos": int(y.sum()), "n_neg": int((1 - y).sum())})
    return pd.DataFrame(rows)


def macro_auc(table):
    """Challenge metric: mean of per-target AUCs; undefined targets excluded and counted."""
    return float(table["auc"].mean(skipna=True)), int(table["auc"].notna().sum())

SCORE = {"positive": 0.9, "not_mentioned": 0.3, "negative": 0.1, "": 0.3}
preds = pd.DataFrame([json.loads(l) for l in open(run_dir / "predictions.jsonl")])
preds[["StudyInstanceUID", "target_name"]] = preds["example_id"].str.split("::", expand=True)
llm = (preds.assign(score=preds["output"].map(SCORE))
       .pivot(index="StudyInstanceUID", columns="target_name", values="score")[TARGET_COLUMNS])

weak = pd.read_csv(find_input("weak_labels_v1.csv"), dtype={"StudyInstanceUID": str})
weak = weak[weak["is_gold"].eq(1)].set_index("StudyInstanceUID").loc[llm.index]
gold_y = weak[[t + "__gold" for t in TARGET_COLUMNS]].set_axis(TARGET_COLUMNS, axis=1).astype(int)
regex = weak[[t + "__weak" for t in TARGET_COLUMNS]].set_axis(TARGET_COLUMNS, axis=1)

llm_t, regex_t = auc_table(gold_y, llm), auc_table(gold_y, regex)
cmp = pd.DataFrame({
    "target": TARGET_COLUMNS,
    "llm_auc": llm_t["auc"], "regex_auc": regex_t["auc"],
    "llm_not_mentioned": [(preds.loc[preds["target_name"].eq(t), "output"] == "not_mentioned").mean()
                          for t in TARGET_COLUMNS],
    "regex_silent": [weak[t + "__silent"].mean() for t in TARGET_COLUMNS],
})
display(cmp.round(3))
(lm, ln), (rm, rn) = macro_auc(llm_t), macro_auc(regex_t)
print(f"macro AUC  LLM {lm:.4f} ({ln}/12) | regex {rm:.4f} ({rn}/12) | diff {lm - rm:+.4f}")
cmp.to_csv(OUT / "llm_vs_regex_v1.csv", index=False)